##ML Guitar Effect Classification Project
CMPU-395: Machine Learning

Axel Estrada




This file contains all of the code used for my ML Final Project. It's sectioned off into the following parts:


*   Dataset Generation and Loading
*   HDF5 Shard Generation
*   Creating Train/Test Split
*   Creating the Model
*   Training and Evaluation
*   Extra code for generating more Mel-Spectrograms

Each cell will detail whether or not it needs to be run in order to properly train and/or evaluate a model.

In general, if you're not wanting to make your own files and don't have the original .wav/mel-specs, any code that involves creating shards can be skipped, as a sample shard dataset is already provided.

For the provided sample dataset, make sure that you upload it to '/contents' and that all the directories are properly named and organized. (NOTE: I will either add some function for downloading the sample dataset from the project github, or for reorganizing all the files into the correct folders if they are uploaded straight to Colab).

In [ ]:
# @title Import (RUN THIS FIRST)

import torch
import torchaudio

from torch import nn
from torch.cuda import is_available
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

import torchvision.models as models
from torchvision import transforms
from torchvision import datasets
from torchsummary import summary

import numpy as np
import pandas as pd

import os
import h5py
import random
import math

from pathlib import Path
import matplotlib.pyplot as plt # for visualizing mel spectrograms
from tqdm import tqdm    # for my own sanity
import librosa


In [ ]:
# @title Google Drive Import (Don't run unless you're me)
# This part is for loading the shards from my synced google drive
from google.colab import files
from google.colab import drive

drive.mount('/content/drive')
!cp -r "/content/drive/Othercomputers/My Laptop/hdf5_shards" /content/

In [ ]:
# @title Global Constants (Always Run)

# MEL-SPECTOGRAM GENERATOR
SAMPLE_RATE = 22050 # sampling rate in Hz
N_MELS=128          # number of mel bands to generate (i.e., the height of the mel spectrogram)
WINDOW_LENGTH=1024  # fft window size in samples (fft stands for "fast fourier transform" - this is the number of samples used to compute each column of the spectrogram)
HOP_SIZE=512        # hop length in samples

# FILE LOCATIONS
# Note: Adjust these paths as needed for your local setup or Colab environment
BASE_DIR = Path.cwd()
DATASETS_DIR = BASE_DIR / "Guitar_FX_Classification_Project"

MEL_SPECTROGRAMS_DIR = Path(DATASETS_DIR) / "Mel_Spectrograms"
CLEAN_RECORDINGS_DIR = Path(MEL_SPECTROGRAMS_DIR) / "Clean_Recordings_Mel_Specs" # Note: for .npy files generated from .wav files

BASE_SHARDS_DIR = Path(DATASETS_DIR) / "hdf5_shards"
MERGED_CSV_PATH = Path(DATASETS_DIR) / "merged_metadata.csv"

MEL_FOLDER_NAME = "mel_22050_1024_512"
METADATA_NAME = "proc_settings.csv"

# TRAINING PARAMETERS
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 0.001
SHARD_SIZE = 10000 # for fragmenting HDF5 files into small chunks

EFFECT_MAP = {
    "808": "overdrive",
    "TS9": "overdrive",
    "BD2": "overdrive",
    "OD1": "overdrive",
    "SD1": "overdrive",
    "MGS": "overdrive",

    "DS1": "distortion",
    "RAT": "distortion",
    "DPL": "distortion",
    "MT2": "distortion",

    "FFC": "fuzz",
    "BMF": "fuzz",
    "RBM": "fuzz",
    "VTB": "fuzz"
}

In [ ]:
# @title Colab Directories (Only run in Colab)
# This part assumes you have uploaded the files directly into /contents
BASE_DIR = Path.cwd()
BASE_SHARDS_DIR = BASE_DIR / "hdf5_shards"
MERGED_CSV_PATH = BASE_DIR / "merged_metadata.csv"

---

&nbsp;

# GENERATING AND LOADING DATASETS



**Dataset Snapshot (Borrowed from GUITAR-FX-DIST):**

*   **Size:** \~550k samples (~305 hours) + 550k mel spectrograms

*   **Audio Format:** WAV - 44.1kHz, 16bit, mono, -6dBFS

*   **Mel-Spectrogram Format:** NPY - 128 frequency bands, sample rate 22050Hz, window length 1024, hop size 512,

*   **Effects:** 14 between overdrive, distortion and fuzz

*   **Unprocessed recordings**
    *   624 monophonic notes

    *   420 polyphonic (2, 3 and 4 notes intervals and chords)

    *   2 guitars, with up to 2 pick-up settings and up to 3 plucking styles (finger pluck - hard, finger pluck - soft, pick)
        *   Schecter Diamond C-1 Classic
        *   Chester Stratocaster
        
    *   **Samples length:** 2 sec

## Create Merged Metadata CSV

This cell doesn't need to be run if you already have 'merged_metadata.csv' downloaded.

In [ ]:
# Scans all proc_settings.csv files in each folder and merges them into a single mega CSV
# This is useful for Colab where you can upload one file instead of the entire folder structure

merged_rows = []
for major_folder in Path(MEL_SPECTROGRAMS_DIR).iterdir():
    if not major_folder.is_dir():
        continue

    audio_type = major_folder.name

    for fx_folder in major_folder.iterdir():
        if not fx_folder.is_dir():
            continue

        csv_path = fx_folder / METADATA_NAME
        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)
        df["audio_type"] = audio_type
        df["fx"] = fx_folder.name
        merged_rows.append(df)

# Combine all CSVs into one and then save
merged_df = pd.concat(merged_rows, ignore_index=True)
merged_df.to_csv(MERGED_CSV_PATH, index=False)

print(f"Merged metadata CSV saved to:\n{MERGED_CSV_PATH}")
print(f"# rows: {len(merged_df)}")
print(f"Columns:\n{list(merged_df.columns)}")

In [ ]:
# @title Load Mel-Spectrograms (Always Run)
# Note: This cell loads the merged metadata CSV created in the previous cell.
# If you're working in Colab, just upload the merged_metadata.csv file

df = pd.read_csv(MERGED_CSV_PATH)

# Add mel_path column - construct paths from audio_type, fx, and filename
df["mel_path"] = df.apply(
    lambda row: str(
        Path(MEL_SPECTROGRAMS_DIR) / row["audio_type"] / row["fx"] / MEL_FOLDER_NAME / row["filename"].replace(".wav", ".npy")
    ),
    axis=1
)

# changing fx to match EFFECTS_MAPPING for classification
df["fx"] = df["fx"].map(EFFECT_MAP)
df = df.dropna(subset=["fx"])

# ===== ADD CLEAN RECORDINGS =====
clean_spectrograms = Path(MEL_SPECTROGRAMS_DIR) / "Clean"
mono_clean_files = list((clean_spectrograms / "Mono_Clean").glob("*.npy"))
poly_clean_files = list((clean_spectrograms / "Poly_Clean").glob("*.npy"))

clean_rows = []
for mel_file in mono_clean_files:
    clean_rows.append({
        "audio_type": "Mono_Clean",
        "fx": "clean",
        "mel_path": str(mel_file),
        "filename": mel_file.name
    })

for mel_file in poly_clean_files:
    clean_rows.append({
        "audio_type": "Poly_Clean",
        "fx": "clean",
        "mel_path": str(mel_file),
        "filename": mel_file.name
    })

df_clean = pd.DataFrame(clean_rows)
df = pd.concat([df, df_clean], ignore_index=True)

# adding a label so effects can be indexed as 0, 1, 2, or 3
# Note: This variable is referenced later for classification labels
label_to_idx = {label: i for i, label in enumerate(sorted(df["fx"].unique()))}
df["label"] = df["fx"].map(label_to_idx)

# regrouping based on audio type
# this step lets us generate .HDF5 files seperately
groups = df["audio_type"].unique()
df_audio_dict = {}
for g in groups:
    df_audio_dict[g] = df[df["audio_type"] == g].reset_index(drop=True)

print("Successfully created and updated our initial Data Frame!\n")
print(f"Total samples: {len(df)}")
print(f"Audio types: {list(groups)}")
print(f"Label mapping: {label_to_idx}")

---

&nbsp;

# HDF5 SHARD GENERATION

Don't run unless any of the cells below unless you have the original mel-spectrograms downloaded and properly formatted.

In [ ]:
# @title Function for Building Shards
def build_sharded_hdf5(df,                      # Dataframe
                       output_base_dir,         # base directory to save shards
                       audio_type,              # for naming and grouping shards
                       target_min_shards=None   # minimum number of shards to create (if None, uses fixed SHARD_SIZE)
                       ):
    df = df.reset_index(drop=True)

    # group by fx so we can print + name shards properly
    fx_groups = df.groupby("fx")

    for fx, fx_df in fx_groups:
        print(f"\n=== Processing FX: {fx} ({audio_type}) ===")

        fx_df = fx_df.reset_index(drop=True)

        # Determine shard size based on target_min_shards
        if target_min_shards is not None:
            # Calculate dynamic shard size to achieve target number of shards
            shard_size = max(1, len(fx_df) // target_min_shards)
        else:
            shard_size = SHARD_SIZE

        num_shards = (len(fx_df) + shard_size - 1) // shard_size

        for shard_idx in range(num_shards):
            start = shard_idx * shard_size
            end = min((shard_idx + 1) * shard_size, len(fx_df))

            shard_df = fx_df.iloc[start:end]

            shard_path = f"{output_base_dir}/{audio_type}/{audio_type}_{fx}_shard_{shard_idx:03d}.h5"

            if os.path.exists(shard_path):
                print(f"Skipping existing {shard_path}")
                continue

            if os.path.exists(shard_path):
                print(f"Skipping existing {shard_path}")
                continue

            mel_list = []
            label_list = []
            skipped = 0

            print(f"Adding mel-spectrograms to {audio_type}_{fx}_shard_{shard_idx:03d}...")

            for row in tqdm(shard_df.itertuples(index=False), total=len(shard_df)):
                path = row.mel_path

                # skip missing files
                if not os.path.exists(path):
                    skipped += 1
                    continue

                try:
                    mel = np.load(path).astype(np.float32)
                except Exception:
                    skipped += 1
                    continue

                mel_list.append(mel)
                label_list.append(row.label)

            # skip empty shard
            if len(mel_list) == 0:
                print(f"Skipped empty shard {shard_idx} (all files missing)")
                continue

            # stack into arrays
            mel_array = np.stack(mel_list)
            label_array = np.array(label_list)

            # write clean shard
            with h5py.File(shard_path, "w") as f:
                f.create_dataset("mel", data=mel_array, dtype="float32")
                f.create_dataset("label", data=label_array, dtype="int64")

            print(f"Generated {audio_type}_{fx}_shard_{shard_idx:03d}.h5 "
                  f"({len(mel_array)} samples, skipped {skipped})")

        print(f"--- Finished FX: {fx} ({audio_type}) ---")

    print(f"\nAll shards generated for {audio_type}!")

In [ ]:
# @title Folders for Shards
for audio_type in groups:
    os.makedirs(f"{BASE_SHARDS_DIR}/{audio_type}", exist_ok=True)

In [ ]:
# @title Generate all HDF5 Files
for audio in df_audio_dict:
    # Use smaller shards for clean recordings to ensure 6-8 shards each
    target_shards = 6 if "Clean" in audio else None

    print(f"Generating shards for {audio}")
    build_sharded_hdf5(
        df=df_audio_dict[audio],
        output_base_dir=BASE_SHARDS_DIR,
        audio_type=audio,
        target_min_shards=target_shards
    )

---

&nbsp;

# CREATING TRAIN/TEST SPLITS

In [ ]:
# @title Train/Test Split Function (Always Run)

# Scans all HDF5 files across all folders and creates train/test split.
# Returns lists of file paths for training and testing.
def create_hdf5_train_test_split(hdf5_base_dir, test_size, random_state):

    random.seed(random_state)
    hdf5_files = []
    base_path = Path(hdf5_base_dir)

    # Scan all subdirectories for HDF5 files
    for audio_type_folder in base_path.iterdir():
        if audio_type_folder.is_dir():
            for h5_file in audio_type_folder.glob("*.h5"):
                hdf5_files.append(str(h5_file))

    print(f"\n=== Creating HDF5 Train/Test Split ===")
    print(f"Total HDF5 files found: {len(hdf5_files)}")

    # Shuffle and split
    random.shuffle(hdf5_files)
    split_idx = int(len(hdf5_files) * (1 - test_size))

    train_files = hdf5_files[:split_idx]
    test_files = hdf5_files[split_idx:]

    # high key remove all the print statements and just return the lists of files
    print(f"Training files: {len(train_files)} ({len(train_files) / len(hdf5_files) * 100:.1f}%)")
    print(f"Testing files: {len(test_files)} ({len(test_files) / len(hdf5_files) * 100:.1f}%)")

    # Print breakdown by category
    print(f"\n--- Files by Category ---")
    categories = {}
    for file in hdf5_files:
        # Extract category from filename (e.g., "Mono_Continuous_distortion_shard_000.h5")
        parts = file.split("_shard_")[0].split(os.sep)[-1]
        categories[parts] = categories.get(parts, 0) + 1

    for cat, count in sorted(categories.items()):
        print(f"{cat}: {count}")

    print(f"--- End Split ---\n")

    return train_files, test_files

In [ ]:
# @title Creating Our Splits (Always Run)
# Note: The split percentage will vary based on the number of HDF5 files uploaded,
# but should roughly fit the percentage set
train_files, test_files = create_hdf5_train_test_split(
    hdf5_base_dir=BASE_SHARDS_DIR,
    test_size=0.2,
    random_state=43
)

In [ ]:
# @title Spectrogram Dataset Class (Always Run)

# Dataset class that handles multiple HDF5 files.
# This is used for processing the mel spectrograms and labels during training and testing.
class Spectrogram_Dataset(Dataset):

    def __init__(self, h5_file_list):
        self.h5_files = h5_file_list
        self.file_info = []  # List of (file_path, num_samples)
        self.total_samples = 0

        # Pre-scan files to know how many samples each contains
        for h5_file in self.h5_files:
            try:
                with h5py.File(h5_file, "r") as f:
                    num_samples = len(f["mel"])
                    self.file_info.append((h5_file, num_samples))
                    self.total_samples += num_samples

            except Exception as e:
                print(f"Warning: Could not read {h5_file}: {e}")

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        # Find which file this index belongs to
        cumulative = 0
        for h5_file, num_samples in self.file_info:
            if idx < cumulative + num_samples:
                local_idx = idx - cumulative

                # Load from file
                with h5py.File(h5_file, "r") as f:
                    x = f["mel"][local_idx].astype(np.float32)
                    y = f["label"][local_idx]

                # Preprocess mel spectrogram
                x = np.log1p(x) # log scaling; compresses range so that louder sounds don't dominate
                x = (x - x.mean()) / (x.std() + 1e-6) # Normalization; zero mean and unit variance

                # Convert to tensors
                x = torch.tensor(x).float().unsqueeze(0)  # Add channel dimension
                y = torch.tensor(y).long()

                return x, y

            cumulative += num_samples

        raise IndexError(f"Index {idx} out of range for dataset with {self.total_samples} samples")


In [ ]:
# @title Create DataLoaders from HDF5 Files (Always Run)
# This code generates the actual splits
train_dataset = Spectrogram_Dataset(train_files)
test_dataset = Spectrogram_Dataset(test_files)

print(f"Train dataset size: {len(train_dataset)} samples")
print(f"Test dataset size: {len(test_dataset)} samples")

train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True,     # randomizing ordering
                          num_workers=0,    # set to 0 for HDF5 (compatibility issue)
                          pin_memory=True)  # speeds up transfer to GPU if True

test_loader = DataLoader(test_dataset,
                         batch_size=BATCH_SIZE,
                         shuffle=False,
                         num_workers=0,
                         pin_memory=True)

MODEL_SAVE_PATH = "guitar_fx_classifier_ResNet.pth"



---

&nbsp;

In [ ]:
# @title CNN Model (VGG)
# This model is inspired by the VGG architecture, which is known for its simplicity and effectiveness in image classification tasks.
# Note: This is a smaller version of VGG to fit our dataset and computational constraints.
# The architecture consists of:
# - 4 convolutional blocks (Conv2d + ReLU + MaxPool2d)
# - Adaptive average pooling to ensure consistent output size regardless of input dimensions
# - Flattening and fully connected layers for classification

class simpleVGG(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        # 4 conv blocks / flatten / linear / softmax
        self.conv1 = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=2
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                stride=1,
                padding=2
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                stride=1,
                padding=2
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                stride=1,
                padding=2
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        # Adaptive pooling to ensure consistent output size
        # regardless of input dimensions
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        # flatten and linear layers for classification
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),  # 128 channels * 4x4 spatial dims
            nn.ReLU(),
            nn.Linear(256, num_classes)
            )

    def forward(self, input_data):
        x = self.conv1(input_data)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.adaptive_pool(x)  # Normalize spatial dimensions to 4x4
        logits = self.classifier(x)

        # Don't need if we're using Cross_Entropy_Loss()
        # predictions = self.softmax(logits)
        return logits

# creating model
MODEL = simpleVGG(num_classes=len(EFFECT_MAP))
MODEL_SAVE_PATH = "guitar_fx_classifier_simpleVGG_4classes.pth"


In [ ]:
# @title ResNet18 Model for Mel-Spectrograms
MODEL = models.resnet18(pretrained=True)

# Modify first convolution to accept 1-channel input (mel-spectrograms) instead of 3 (RGB)
MODEL.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Replace final FC layer for guitar effect classification (now 4 classes: overdrive, distortion, fuzz, clean)
MODEL.fc = nn.Linear(MODEL.fc.in_features, len(label_to_idx))

MODEL_SAVE_PATH = "guitar_fx_classifier_ResNet.pth"

In [ ]:
# @title Instantiate Loss Function + Optimizer (Always Run)
loss_fn = nn.CrossEntropyLoss() # remember that this already uses softmax
optimizer = torch.optim.Adam(MODEL.parameters(),
                             lr=LEARNING_RATE)

# Setting up model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = MODEL.to(device)
print(f"Using {device} device")


---

&nbsp;

In [ ]:
# @title Model Training Functions (Always Run)

def train_one_epoch(model, train_loader, loss_fn, optimizer, device, epoch_num):
    """
    Train for one epoch using DataLoader
    """
    model.train()  # Set model to training mode
    total_samples = 0
    epoch_loss = 0.0

    print(f"\n--- Epoch {epoch_num} Training ---")

    # Progress bar for batches
    with tqdm(total=len(train_loader), desc=f"Epoch {epoch_num}", unit="batch") as pbar:
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            # Move data to device
            inputs = inputs.to(device)
            targets = targets.to(device)

            # Forward pass
            predictions = model(inputs)
            loss = loss_fn(predictions, targets)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Track metrics
            epoch_loss += loss.item()
            total_samples += inputs.size(0)

            # Update progress bar
            pbar.update(1)
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = epoch_loss / max(len(train_loader), 1)
    print(f"Epoch {epoch_num} - Avg Loss: {avg_loss:.4f} | Total Samples: {total_samples}")

# Full Training Loop
def train(model, train_loader, loss_fn, optimizer, device, n_epochs):
    """
    Full training loop with model parameters and progress tracking.
    """
    print("\n" + "="*50)
    print("MODEL TRAINING STARTED")
    print("="*50)

    # Print model parameters
    total_params = sum(p.numel() for p in model.parameters())

    print(f"\n--- Model Parameters ---")
    print(f"Total Parameters: {total_params:,}")
    print(f"Device: {device}")
    print(f"Batch Size: {BATCH_SIZE}")
    print(f"Learning Rate: {LEARNING_RATE}")
    print(f"Optimizer: {optimizer.__class__.__name__}")
    print(f"Loss Function: {loss_fn.__class__.__name__}")
    print(f"Total Epochs: {n_epochs}")
    print(f"Total Batches per Epoch: {len(train_loader)}")

    print("\n" + "="*50 + "\n")

    # Training loop
    for epoch in range(n_epochs):
        train_one_epoch(model, train_loader, loss_fn, optimizer, device, epoch + 1)
        print("-" * 50)

    print("\n" + "="*50)
    print("Training Completed!")
    print("="*50 + "\n")

In [ ]:
# @title Load Model if it Exists (Optional)
if os.path.exists(MODEL_SAVE_PATH):
    print(f"Loading existing model from {MODEL_SAVE_PATH}...")
    checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device)
    #MODEL.load_state_dict(checkpoint["state_dict"])

    print("Model info:")
    model_params = checkpoint.get("model_params", {})

    labels = model_params.get("label_to_idx", label_to_idx)
    effects_map = model_params.get("effect_map", EFFECT_MAP)
    architecture = model_params.get("architecture", None)
    epoch = checkpoint.get("epoch", "unknown")

    print(f"  labels: {labels}")
    print(f"  effects map: {effects_map}")
    print(f"  epochs: {epoch}")

    if architecture is not None:
        print(f"  architecture: {architecture}")

    total_params = sum(p.numel() for p in MODEL.parameters())
    print(f"  total parameters: {total_params:,}")


In [ ]:
# @title Finally, we can start training our model!
train(MODEL, train_loader, loss_fn, optimizer, device, EPOCHS)

In [ ]:
# @title Saving Our Model (Optional)

# Create a checkpoint with model state and metadata
checkpoint = {
    'epoch': EPOCHS,
    'state_dict': MODEL.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss_fn,
    'model_params': {
        'num_classes': len(EFFECT_MAP),
        'label_to_idx': label_to_idx,
        'effect_map': EFFECT_MAP,
        'architecture': 'Simple-VGG',
    }
}

# Save the checkpoint
torch.save(checkpoint, MODEL_SAVE_PATH)
print(f"Model checkpoint saved to {MODEL_SAVE_PATH}")

In [ ]:
# @title Model Evaluation Function

def evaluate(model, test_loader, device):
    """
    Evaluate model on the entire test set and return accuracy.
    """
    model.eval()
    correct = 0
    total = 0

    print(f"\n--- Evaluating on Test Set ---")

    with torch.no_grad(): # no need to track gradients during evaluation
        with tqdm(total=len(test_loader), desc="Evaluating", unit="batch") as pbar:
            for inputs, targets in test_loader:
                inputs = inputs.to(device)
                targets = targets.to(device)

                # Get predictions
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)

                # Count correct predictions
                total += targets.size(0)
                correct += (predicted == targets).sum().item()

                pbar.update(1)

    accuracy = 100 * correct / total
    print(f"\n==== Test Set Results ====")
    print(f"Total Samples: {total}")
    print(f"Correct Predictions: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"=========================\n")

    return accuracy

In [ ]:
# @title Run Evaluation
test_accuracy = evaluate(MODEL, test_loader, device)

In [ ]:
# @title Visualizing Model Predictions (Optional)

# Create reverse mapping from index to label
idx_to_label = {v: k for k, v in label_to_idx.items()}

# This function takes random test samples and then plots the mel-spectrograms
# Above the graphs are the labels for classification and what the model predicted
# (Red indiciates incorrect; Green indiciates correct)
def visualize_predictions(model, test_dataset, device, num_samples=4):

    # Calculate grid dimensions dynamically
    cols = min(2, num_samples)  # Default to 2 columns, or fewer if num_samples < 2
    rows = math.ceil(num_samples / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5 * rows))

    # Handle case where there's only 1 subplot (axes won't be an array)
    if num_samples == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    model.eval()

    # Get random training indices
    random_indices = np.random.choice(len(test_dataset), num_samples, replace=False)

    with torch.no_grad():
        for idx, data_idx in enumerate(random_indices):
            # Get sample
            mel_spec, true_label = test_dataset[data_idx]

            # Add batch dimension (i.e. shape becomes [1, 128, time_frames])
            mel_spec_batch = mel_spec.unsqueeze(0).to(device)

            # Get prediction
            output = model(mel_spec_batch)
            _, predicted_label = torch.max(output, 1)

            predicted_label_indices = predicted_label.item()
            true_label_indices = true_label.item()

            # Convert indices to class names
            true_class = idx_to_label[true_label_indices]
            predicted_class = idx_to_label[predicted_label_indices]

            # Plot spectrogram using librosa (remove channel dimension and convert to numpy)
            mel_np = mel_spec.squeeze(0).cpu().numpy()

            # Display mel-spectrogram with proper frequency and time axes
            im = librosa.display.specshow(
                mel_np,
                sr=SAMPLE_RATE,
                hop_length=HOP_SIZE,
                x_axis='time',
                y_axis='mel',
                ax=axes[idx],
                cmap='viridis'
            )

            # Color the title based on correctness
            is_correct = (true_label_indices == predicted_label_indices)
            title_color = 'green' if is_correct else 'red'

            axes[idx].set_title(
                f"True: {true_class}\nPredicted: {predicted_class}",
                fontsize=12,
                color=title_color,
                fontweight='bold'
            )

            # Add colorbar for illustrating intensity of mel spectrogram (in decibels)
            cbar = fig.colorbar(im, ax=axes[idx], format='%+2.0f dB')

    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize 4 random samples
visualize_predictions(MODEL, test_dataset, device, num_samples=6)

---

&nbsp;

## Extra Data Generation for Further Analysis (All Optional)

The code below is for generating new Mel Spectrograms from .wav files. This code was primarily used for generating spectrograms for the clean, pre-processed datasets.

In [ ]:
# @title Data Loader Class for Generating Mel-Spectograms
# Simplified processor for converting WAV files to mel spectrograms.
# Specifically designed for batch processing of Clean_Recordings folders.
class Audio_To_Spectrogram_Processor:

    def __init__(self,
                 target_sample_rate=SAMPLE_RATE,  # Sample rate to resample to
                 num_samples=SAMPLE_RATE * 2):    # Number of samples to normalize audio to

        self.target_sample_rate = target_sample_rate
        self.num_samples = num_samples

        # Setup mel spectrogram transform to match datasets format
        # 128 frequency bands, window length 1024, hop size 512
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.target_sample_rate,
            n_mels=N_MELS,
            n_fft=WINDOW_LENGTH,
            hop_length=HOP_SIZE)

    """HELPER FUNCTIONS FOR PREPROCESSING"""

    def _resample_if_necessary(self, signal, sr):
        """Resample signal to target sample rate if necessary."""

        if sr != self.target_sample_rate:
            resampler = torchaudio.transforms.Resample(sr, self.target_sample_rate)
            signal = resampler(signal)
        return signal

    def _mix_down_if_necessary(self, signal):
        """Convert multi-channel signal to mono if necessary."""

        if signal.shape[0] > 1:
            signal = torch.mean(signal, dim=0, keepdim=True)
        return signal

    def _cut_if_necessary(self, signal):
        """Trim signal if it's longer than num_samples."""

        if signal.shape[1] > self.num_samples:
            signal = signal[:, :self.num_samples]
        return signal

    def _right_pad_if_necessary(self, signal):
        """Pad signal with zeros if it's shorter than num_samples."""

        length_signal = signal.shape[1]
        if length_signal < self.num_samples:
            num_missing_samples = self.num_samples - length_signal
            last_dim_padding = (0, num_missing_samples)
            signal = torch.nn.functional.pad(signal, last_dim_padding)
        return signal

    # Main function to process a single audio file and return mel spectrogram
    # Returns: 2D numpy array (freq_bins x time_frames) or None if error
    def process_audio_file(self,
                           wav_file_path):

        try:
            # Load audio file
            waveform, sr = librosa.load(wav_file_path, sr=self.target_sample_rate)
            waveform = torch.tensor(waveform).unsqueeze(0)

            # Apply preprocessing steps
            waveform = self._resample_if_necessary(waveform, sr)
            waveform = self._mix_down_if_necessary(waveform)
            waveform = self._cut_if_necessary(waveform)
            waveform = self._right_pad_if_necessary(waveform)

            # Generate mel spectrogram
            mel_spec = self.mel_transform(waveform)

            # Convert to numpy and squeeze to 2D (freq_bins, time_frames)
            mel_spec_np = mel_spec.squeeze(0).numpy().astype(np.float32)

            return mel_spec_np

        except Exception as e:
            print(f"Error processing {wav_file_path.name}: {e}")
            return None

    # returns count of successfully processed files
    def process_folder(self,
                       input_dir,           # Path to folder containing WAV files
                       output_dir,          # Path to folder where NPY files will be saved
                       audio_type_name):    # Name for logging (e.g., "Mono (No FX)")

        if not input_dir.exists():
            print(f"Warning: {input_dir} does not exist. Skipping {audio_type_name}.")
            return 0

        skipped = 0
        successful = 0
        wav_files = list(input_dir.glob("*.wav"))

        print(f"\n=== Processing {audio_type_name} ===")
        print(f"Found {len(wav_files)} WAV files")

        for wav_file in tqdm(wav_files, desc=f"Processing {audio_type_name}", unit="file"):
            mel_spec_np = self.process_audio_file(wav_file)

            if mel_spec_np is not None:
                # Save as NPY file with same name as WAV
                output_file = output_dir / wav_file.stem
                np.save(str(output_file), mel_spec_np)
                successful += 1
            else:
                skipped += 1

        print(f"Completed {audio_type_name}: {successful} successful, {skipped} skipped")
        return successful


In [ ]:
# @title Generate Mel-Spectrograms from Clean Recordings

# Initialize the audio processor
audio_processor = Audio_To_Spectrogram_Processor()

# Setup paths
mono_input_dir = Path(DATASETS_DIR) / "Clean_Recordings" / "_NoFX_mono_preprocessed"
poly_input_dir = Path(DATASETS_DIR) / "Clean_Recordings" / "_NoFX_poly_preprocessed"

# Output directories
output_base_dir = Path(MEL_SPECTROGRAMS_DIR) / "Clean"
mono_output_dir = output_base_dir / "Mono_Clean"
poly_output_dir = output_base_dir / "Poly_Clean"

# Create output directories if they don't exist
mono_output_dir.mkdir(parents=True, exist_ok=True)
poly_output_dir.mkdir(parents=True, exist_ok=True)

# Process recordings
mono_count = audio_processor.process_folder(mono_input_dir, mono_output_dir, "Mono_Clean")
poly_count = audio_processor.process_folder(poly_input_dir, poly_output_dir, "Poly_Clean")

print(f"\n{'='*50}")
print(f"Mel Spectrogram Generation Complete!")
print(f"{'='*50}")
print(f"Total Mono Specs Generated: {mono_count}")
print(f"Total Poly Specs Generated: {poly_count}")
print(f"Output Location: {output_base_dir}")